# Generic SAP HANA to Foundation layer Ingestion

Reads SAP HANA tables over JDBC, adds audit columns and writes them to the foundation layer, driven by `ict_metadata.etl_setup_config` (rows with `target_layer = 'foundation'` and `is_active = true`, in `load_sequence` order).

* Per entity, the config gives the HANA table (`source_read_from`), target table (`target_table`), write mode (`load_type`: `append` or `overwrite`) and `partition_column`.
* Host, port, schema, user and password come from the Key Vault secret scope given by the `secret_scope` widget (set per environment by the bundle; dev default `coentictaius6kvdevtst01`).
* JDBC URL: `jdbc:sap://{server_name}:{port}?currentSchema={schema}`.
* Widgets: `catalog` (default `ent_gf_dev`), `secret_scope` (default `coentictaius6kvdevtst01`), `entity_ids` (comma-separated; blank = all active foundation entities).
* Each entity writes a SUCCESS/FAILED row to `ict_audit.audit_task_run`.

remove stable_part function, fetch it from dbr workflow


In [0]:
%run /Shared/gskglobal-ict/bundle_ict/Utilities/Audit_Utils

In [0]:
%python
from pyspark.sql import functions as F

                  

dbutils.widgets.text("catalog", "ent_gf_dev", "Catalog")
dbutils.widgets.text("secret_scope", "coentictaius6kvdevtst01", "Secret scope")
dbutils.widgets.text("entity_ids", "",)
catalog = dbutils.widgets.get("catalog").strip()
SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
entity_ids = [e.strip() for e in dbutils.widgets.get("entity_ids").split(",") if e.strip()]
AUDIT_TASK_TABLE = f"{catalog}.ict_audit.audit_task_run"
WATERMARK_TABLE = f"{catalog}.ict_metadata.etl_watermark"

# Foundation load config
config_df = spark.table(f"{catalog}.ict_metadata.etl_setup_config").filter(
    (F.col("target_layer") == "foundation") & F.col("is_active")
)
if entity_ids:
    config_df = config_df.filter(F.col("entity_id").isin(entity_ids))
configs = config_df.orderBy("load_sequence").collect()
if not configs:
    raise ValueError(f"No active foundation config found for entity_ids={entity_ids or 'ALL'}")

# Connection details from Key Vault
if SECRET_SCOPE not in [s.name for s in dbutils.secrets.listScopes()]:
    raise ValueError(f"Secret scope '{SECRET_SCOPE}' not found or not accessible")
server_name = dbutils.secrets.get(scope=SECRET_SCOPE, key="hana-host")
port = dbutils.secrets.get(scope=SECRET_SCOPE, key="hana-port")
schema = dbutils.secrets.get(scope=SECRET_SCOPE, key="hana-source-schema")
user_name = dbutils.secrets.get(scope=SECRET_SCOPE, key="hana-user-name")
password = dbutils.secrets.get(scope=SECRET_SCOPE, key="hana-password")

jdbc_url = f"jdbc:sap://{server_name}:{port}?currentSchema={schema}"
for c in configs:
    print(f"{c['entity_id']}: {schema}.{c['source_read_from']} -> {catalog}.{c['target_table']} ({c['load_type']})")

In [0]:
%python
import uuid
from datetime import datetime, UTC
from delta.tables import DeltaTable


def read_hana_table(table_name, where=None):
    where_clause = f" WHERE {where}" if where else ""
    query = f'(SELECT * FROM "{schema}"."{table_name}"{where_clause}) AS src'
    return (
        spark.read.format("jdbc")
        .option("driver", "com.sap.db.jdbc.Driver")
        .option("url", jdbc_url)
        .option("dbtable", query)
        .option("user", user_name)
        .option("password", password)
        .load()
    )


def ingest_entity(cfg, run_id):
    hana_table = cfg["source_read_from"].strip().upper()  # HANA table names are uppercase; quoted identifiers are case-sensitive
    target_table = f"{catalog}.{cfg['target_table']}"
    load_type = cfg["load_type"].lower()

    # Watermark: empty last_load_type = initial (full) load, otherwise incremental on watermark_column
    wm = spark.table(WATERMARK_TABLE).filter(
        (F.col("entity_id") == cfg["entity_id"]) & (F.col("layer") == "foundation")
    ).first()
    wm_col = wm["watermark_column"]
    is_initial = not wm["last_load_type"]
    new_load_type = "initial" if is_initial else "incremental"

    audit_args = dict(
        table_name=cfg["entity_id"], run_id=run_id, layer=cfg["target_layer"], target_table=target_table,
        load_type=load_type.upper(), source_schema=schema, start_ts=datetime.now(UTC),
        watermark_from=wm["last_watermark_value"],
    )
    try:
        if is_initial:
            src_df = read_hana_table(hana_table)
        else:
            last_wm = wm["last_watermark_value"].strftime("%Y%m%d%H%M%S")  
            src_df = read_hana_table(
                hana_table,
                f'"{wm_col}" > {last_wm} AND "{wm_col}" <= (SELECT MAX("{wm_col}") FROM "{schema}"."{hana_table}")',
            )
            if src_df.isEmpty():
                write_task_audit(spark, build_task_audit_details(
                    **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), row_count=0,
                    message=f"No new rows in {schema}.{hana_table} after {wm_col} {last_wm}",
                ), AUDIT_TASK_TABLE)
                print(f"{cfg['entity_id']}: no new rows after {wm_col} {last_wm}, load skipped")
                return {"entity_id": cfg["entity_id"], "status": "SUCCESS", "error": None}

        # Audit columns: one fixed _ingest_ts per run, used below to find this run's rows
        ingest_ts = datetime.now(UTC)
        df = src_df.withColumn("_ingest_ts", F.lit(ingest_ts)).withColumn("_ingest_date", F.to_date(F.col("_ingest_ts")))
        writer = df.write.format("delta").mode(load_type).option("mergeSchema", "true")
        if cfg["partition_column"]:
            writer = writer.partitionBy(cfg["partition_column"])
        writer.saveAsTable(target_table)

        rows_written = DeltaTable.forName(spark, target_table).history(1).first()["operationMetrics"]["numOutputRows"]

        # New watermark = max watermark_column of the rows loaded in this run
        new_wm = (
            spark.table(target_table).filter(F.col("_ingest_ts") == F.lit(ingest_ts))
            .agg(F.max(F.to_timestamp(F.col(wm_col).cast("string"), "yyyyMMddHHmmss"))).first()[0]
        )
        spark.sql(
            f"""UPDATE {WATERMARK_TABLE}
                SET previous_watermark_value = last_watermark_value,
                    last_watermark_value = :new_wm,
                    last_run_id = :last_run_id,
                    last_load_type = :last_load_type,
                    last_success_ts = current_timestamp(),
                    updated_by = current_user()
                WHERE entity_id = :entity_id AND layer = 'foundation'""",
            args={"new_wm": new_wm, "last_run_id": stable_long_id(run_id),
                  "last_load_type": new_load_type, "entity_id": cfg["entity_id"]},
        )

        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="SUCCESS", end_ts=datetime.now(UTC), row_count=rows_written, watermark_to=new_wm,
            message=f"{schema}.{hana_table} -> {target_table}",
        ), AUDIT_TASK_TABLE)
        print(f"{cfg['entity_id']}: {new_load_type} load wrote {rows_written} rows to {target_table}, watermark = {new_wm}")
        return {"entity_id": cfg["entity_id"], "status": "SUCCESS", "error": None}

    except Exception as e:
        error_message = str(e)[:2000]
        write_task_audit(spark, build_task_audit_details(
            **audit_args, status="FAILED", end_ts=datetime.now(UTC),
            message=error_message, rows_updated=None, rows_deleted=None,
        ), AUDIT_TASK_TABLE)
        print(f"{cfg['entity_id']}: FAILED - {e}")
        return {"entity_id": cfg["entity_id"], "status": "FAILED", "error": error_message}

In [0]:
%python
run_id = str(uuid.uuid4())
results = [ingest_entity(cfg, run_id) for cfg in configs]

display(spark.createDataFrame(results, "entity_id STRING, status STRING, error STRING"))

failed = [r["entity_id"] for r in results if r["status"] == "FAILED"]
if failed:
    raise RuntimeError(f"Ingestion failed for {len(failed)} entity(ies): {failed}")